In [0]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retaillake")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.retaillake.raw_files
""")


DataFrame[]

In [0]:
base_path = "/Volumes/workspace/retaillake/raw_files/"

orders = (
    spark.read
    .option("header", True)
    .csv(base_path + "olist_orders_dataset.csv")
)

orders.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [0]:
items = spark.read.option("header", True).csv(
    base_path + "olist_order_items_dataset.csv"
)

products = spark.read.option("header", True).csv(
    base_path + "olist_products_dataset.csv"
)

print("Orders:", orders.count())
print("Order items:", items.count())
print("Products:", products.count())

Orders: 99441
Order items: 112650
Products: 32951


In [0]:
orders.printSchema()
items.printSchema()
products.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_w

In [0]:
from pyspark.sql import functions as F

items_typed = (
    items
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("price", F.col("price").cast("decimal(12,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(12,2)"))
)

items_typed.select(
    "order_id", "order_item_id", "price", "freight_value"
).show(5, truncate=False)


+--------------------------------+-------------+------+-------------+
|order_id                        |order_item_id|price |freight_value|
+--------------------------------+-------------+------+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |58.90 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |239.90|19.93        |
|000229ec398224ef6ca0657da4fc703e|1            |199.00|17.87        |
|00024acbcdf0a6daa1e931b038114c75|1            |12.99 |12.79        |
|00042b26cf59d7ce69dfabb4e55b4fd9|1            |199.90|18.14        |
+--------------------------------+-------------+------+-------------+
only showing top 5 rows


In [0]:
orders_typed = orders.withColumn(
    "order_purchase_timestamp",
    F.to_timestamp(
        "order_purchase_timestamp",
        "yyyy-MM-dd HH:mm:ss"
    )
)

orders_typed.select(
    "order_id", "order_purchase_timestamp"
).show(5, truncate=False)

+--------------------------------+------------------------+
|order_id                        |order_purchase_timestamp|
+--------------------------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|2018-07-24 20:41:37     |
|47770eb9100c2d0c44946d9cf07ec65d|2018-08-08 08:38:49     |
|949d5b44dbf5de918fe9c16f97b45f8a|2017-11-18 19:28:06     |
|ad21c59c0840e6cb83a9ceb5573f8159|2018-02-13 21:18:39     |
+--------------------------------+------------------------+
only showing top 5 rows


In [0]:
orders_typed.select(
    F.count(F.when(
        F.col("order_purchase_timestamp").isNull(), 1
    )).alias("missing_purchase_timestamp")
).show()

items_typed.select(
    F.count(F.when(F.col("order_item_id").isNull(), 1))
        .alias("missing_item_number"),
    F.count(F.when(F.col("price").isNull(), 1))
        .alias("missing_price"),
    F.count(F.when(F.col("freight_value").isNull(), 1))
        .alias("missing_freight")
).show()

+--------------------------+
|missing_purchase_timestamp|
+--------------------------+
|                         0|
+--------------------------+

+-------------------+-------------+---------------+
|missing_item_number|missing_price|missing_freight|
+-------------------+-------------+---------------+
|                  0|            0|              0|
+-------------------+-------------+---------------+



Joining the datasets

In [0]:
sales_items = (
    items_typed
    .join(
        orders_typed.select(
            "order_id",
            "order_status",
            "order_purchase_timestamp"
        ),
        on="order_id",
        how="left"
    )
    .join(
        products.select("product_id", "product_category_name"),
        on="product_id",
        how="left"
    )
)

print("Rows after joins:", sales_items.count())
sales_items.select(
    "order_id", "order_status", "product_category_name", "price"
).show(5, truncate=False)

Rows after joins: 112650
+--------------------------------+------------+---------------------+------+
|order_id                        |order_status|product_category_name|price |
+--------------------------------+------------+---------------------+------+
|00010242fe8c5a6d1ba2dd792cb16214|delivered   |cool_stuff           |58.90 |
|00018f77f2f0320c557190d7a144bdd3|delivered   |pet_shop             |239.90|
|000229ec398224ef6ca0657da4fc703e|delivered   |moveis_decoracao     |199.00|
|00024acbcdf0a6daa1e931b038114c75|delivered   |perfumaria           |12.99 |
|00042b26cf59d7ce69dfabb4e55b4fd9|delivered   |ferramentas_jardim   |199.90|
+--------------------------------+------------+---------------------+------+
only showing top 5 rows


In [0]:
delivered_items = (
    sales_items
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "category",
        F.when(
            F.col("product_category_name").isNull()
            | (F.trim(F.col("product_category_name")) == ""),
            "Unknown"
        ).otherwise(F.col("product_category_name"))
    )
)

category_sales = (
    delivered_items
    .groupBy("category")
    .agg(
        F.count("*").alias("items_sold"),
        F.countDistinct("order_id").alias("orders"),
        F.sum("price").alias("item_sales")
    )
    .orderBy(F.desc("item_sales"))
)

category_sales.show(10, truncate=False)

+----------------------+----------+------+----------+
|category              |items_sold|orders|item_sales|
+----------------------+----------+------+----------+
|beleza_saude          |9465      |8647  |1233131.72|
|relogios_presentes    |5859      |5495  |1166176.98|
|cama_mesa_banho       |10953     |9272  |1023434.76|
|esporte_lazer         |8431      |7530  |954852.55 |
|informatica_acessorios|7644      |6530  |888724.61 |
|moveis_decoracao      |8160      |6307  |711927.69 |
|utilidades_domesticas |6795      |5743  |615628.69 |
|cool_stuff            |3718      |3559  |610204.10 |
|automotivo            |4140      |3810  |578966.65 |
|brinquedos            |4030      |3804  |471286.48 |
+----------------------+----------+------+----------+
only showing top 10 rows


Verification

In [0]:
detail_total = delivered_items.agg(
    F.sum("price").alias("total")
).first()["total"]

summary_total = category_sales.agg(
    F.sum("item_sales").alias("total")
).first()["total"]

print("Item-level total:", detail_total)
print("Category summary total:", summary_total)
print("Totals match:", detail_total == summary_total)

Item-level total: 13221498.11
Category summary total: 13221498.11
Totals match: True


In [0]:
category_sales.createOrReplaceTempView("category_sales_result")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.retaillake.category_sales
USING DELTA
AS SELECT * FROM category_sales_result
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT *
FROM workspace.retaillake.category_sales
ORDER BY item_sales DESC
LIMIT 10
""").show(truncate=False)

+----------------------+----------+------+----------+
|category              |items_sold|orders|item_sales|
+----------------------+----------+------+----------+
|beleza_saude          |9465      |8647  |1233131.72|
|relogios_presentes    |5859      |5495  |1166176.98|
|cama_mesa_banho       |10953     |9272  |1023434.76|
|esporte_lazer         |8431      |7530  |954852.55 |
|informatica_acessorios|7644      |6530  |888724.61 |
|moveis_decoracao      |8160      |6307  |711927.69 |
|utilidades_domesticas |6795      |5743  |615628.69 |
|cool_stuff            |3718      |3559  |610204.10 |
|automotivo            |4140      |3810  |578966.65 |
|brinquedos            |4030      |3804  |471286.48 |
+----------------------+----------+------+----------+



In [0]:
monthly_sales = (
    delivered_items
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .groupBy("purchase_month")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.count("*").alias("items_sold"),
        F.sum("price").alias("item_sales")
    )
    .orderBy("purchase_month")
)

monthly_sales.show(30, truncate=False)


+--------------+------+----------+----------+
|purchase_month|orders|items_sold|item_sales|
+--------------+------+----------+----------+
|2016-09       |1     |3         |134.97    |
|2016-10       |265   |313       |40325.11  |
|2016-12       |1     |1         |10.90     |
|2017-01       |750   |913       |111798.36 |
|2017-02       |1653  |1858      |234223.40 |
|2017-03       |2546  |2897      |359198.85 |
|2017-04       |2303  |2569      |340669.68 |
|2017-05       |3546  |4004      |489338.25 |
|2017-06       |3135  |3489      |421923.37 |
|2017-07       |3872  |4416      |481604.52 |
|2017-08       |4193  |4797      |554699.70 |
|2017-09       |4150  |4737      |607399.67 |
|2017-10       |4478  |5214      |648247.65 |
|2017-11       |7289  |8475      |987765.37 |
|2017-12       |5513  |6187      |726033.19 |
|2018-01       |7069  |8037      |924645.00 |
|2018-02       |6555  |7518      |826437.13 |
|2018-03       |7003  |8017      |953356.25 |
|2018-04       |6798  |7827      |

In [0]:
monthly_total = monthly_sales.agg(
    F.sum("item_sales").alias("total")
).first()["total"]

print("Monthly total:", monthly_total)
print("Item-level total:", detail_total)
print("Totals match:", monthly_total == detail_total)

Monthly total: 13221498.11
Item-level total: 13221498.11
Totals match: True


In [0]:
monthly_sales.createOrReplaceTempView("monthly_sales_result")

spark.sql("""
CREATE TABLE workspace.retaillake.monthly_sales
USING DELTA
AS SELECT * FROM monthly_sales_result
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Check if saved

In [0]:
spark.table("workspace.retaillake.monthly_sales") \
    .orderBy("purchase_month") \
    .show(30, truncate=False)

+--------------+------+----------+----------+
|purchase_month|orders|items_sold|item_sales|
+--------------+------+----------+----------+
|2016-09       |1     |3         |134.97    |
|2016-10       |265   |313       |40325.11  |
|2016-12       |1     |1         |10.90     |
|2017-01       |750   |913       |111798.36 |
|2017-02       |1653  |1858      |234223.40 |
|2017-03       |2546  |2897      |359198.85 |
|2017-04       |2303  |2569      |340669.68 |
|2017-05       |3546  |4004      |489338.25 |
|2017-06       |3135  |3489      |421923.37 |
|2017-07       |3872  |4416      |481604.52 |
|2017-08       |4193  |4797      |554699.70 |
|2017-09       |4150  |4737      |607399.67 |
|2017-10       |4478  |5214      |648247.65 |
|2017-11       |7289  |8475      |987765.37 |
|2017-12       |5513  |6187      |726033.19 |
|2018-01       |7069  |8037      |924645.00 |
|2018-02       |6555  |7518      |826437.13 |
|2018-03       |7003  |8017      |953356.25 |
|2018-04       |6798  |7827      |

In [0]:
display(
    spark.table("workspace.retaillake.monthly_sales")
    .orderBy("purchase_month")
)

purchase_month,orders,items_sold,item_sales
2016-09,1,3,134.97
2016-10,265,313,40325.11
2016-12,1,1,10.90
2017-01,750,913,111798.36
2017-02,1653,1858,234223.40
2017-03,2546,2897,359198.85
2017-04,2303,2569,340669.68
2017-05,3546,4004,489338.25
2017-06,3135,3489,421923.37
2017-07,3872,4416,481604.52


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.table("workspace.retaillake.category_sales")
    .orderBy(F.desc("item_sales"))
    .limit(10)
)

category,items_sold,orders,item_sales
beleza_saude,9465,8647,1233131.72
relogios_presentes,5859,5495,1166176.98
cama_mesa_banho,10953,9272,1023434.76
esporte_lazer,8431,7530,954852.55
informatica_acessorios,7644,6530,888724.61
moveis_decoracao,8160,6307,711927.69
utilidades_domesticas,6795,5743,615628.69
cool_stuff,3718,3559,610204.10
automotivo,4140,3810,578966.65
brinquedos,4030,3804,471286.48
